# 实验一：AI 驱动的网络流量分类（Network Traffic Classification）
## FMI Course · Kaggle Hands-on Lab

**课程**：未来媒体互联网（Future Media & Internet） &nbsp;|&nbsp; **预计时长**：约 10–15 分钟 &nbsp;|&nbsp; **运行环境**：Kaggle Notebook（CPU） &nbsp;|&nbsp; **无需 GPU / 无需联网**

---

> **核心问题**：当 Web、视频、下载等应用都可能使用 **443 端口**时，我们还能只靠端口号识别应用吗？

## 实验概述

网络流量分类（Network Traffic Classification）是网络管理中的基础任务：判断一条网络流属于 Web 浏览、DNS 查询、视频流媒体还是文件下载。分类结果可用于 QoS、容量规划、安全分析和网络优化。

早期网络常依赖**端口号（Port Number）**识别应用，例如 HTTP→80、DNS→53、FTP→21。但在现代网络中，HTTPS、QUIC、端口复用和随机端口使这种规则越来越不可靠：**多个应用可能共享同一个端口，尤其是 443。**

本实验用一个小型、可重复的仿真数据集演示两种思路：

- **传统 Baseline**：只依据端口规则判断应用；
- **AI 方法**：忽略端口语义，只使用包长、包间隔、流字节数等统计特征，采用标准化后的 KNN 分类。

通过准确率、二维特征分布和混淆矩阵，你将看到：**当端口发生复用时，行为统计特征比固定端口规则更有区分能力。**

> 本实验用于教学演示，不代表真实网络环境中的最终分类性能。

## 学习目标

完成本实验后，你应该能够：

1. 解释为什么现代网络中仅依赖端口号会失效；
2. 理解包长、包间隔、流字节数等统计特征为什么能够反映应用行为；
3. 用自己的语言描述 K 近邻（KNN）的分类过程；
4. 解释为什么 KNN 前通常需要进行特征标准化；
5. 读懂混淆矩阵，定位最容易相互混淆的流量类别；
6. 区分“教学仿真结果”和“真实网络性能”。

## 背景与基本原理

### 1. 为什么端口号不够用？

传统端口规则非常直观：HTTP 常见 80，DNS 常见 53，FTP 常见 21。但现代网络具有三个典型变化：

- **HTTPS / QUIC 普及**：Web、视频、下载等不同应用都可能使用 443；
- **端口随机化**：部分应用会使用动态高端口；
- **应用复用**：同一传输连接或同一端口上可能承载不同业务语义。

因此，端口仍然是一个可观测的网络字段，但它已经不能稳定代表“应用类型”。

### 2. 流统计特征（Flow Statistical Features）

| 特征 | 含义 | 直觉解释 |
|---|---|---|
| `pkt_len_mean` | 平均包长（B） | DNS 通常较小，视频/下载通常较大 |
| `pkt_len_var` | 包长方差 | 描述包长变化程度 |
| `gap_mean` | 平均包间隔（s） | 持续传输通常间隔更短 |
| `gap_var` | 包间隔方差 | 描述发送节奏是否稳定 |
| `bytes` | 每条流的总字节数 | 下载/视频流通常明显大于 DNS |

这些特征共同构成一条 Flow 在**特征空间（Feature Space）**中的位置。

### 3. K 近邻（K-Nearest Neighbors, KNN）

KNN 的基本思想是：

> 对一条待分类 Flow，寻找训练集中距离最近的 K 条 Flow，用邻居中的多数类别作为预测结果。

本实验使用 **K=5**。

### 4. 为什么要标准化？

KNN 使用距离计算。`bytes` 可能是几十万，而 `gap_mean` 可能只有 0.01。如果直接计算欧氏距离，大数值特征会不成比例地主导结果。

因此本实验使用：

`StandardScaler → KNN`

把不同量纲的统计特征缩放到可比较的尺度后再进行近邻搜索。

### 5. 实验对照

- **Baseline：端口规则**：53→DNS，80/443→Web，21→Download，其余端口记为 Unknown；
- **AI 方法：统计特征 + StandardScaler + KNN**；
- **评价方式**：在同一测试集上比较 Accuracy，并用混淆矩阵分析错误。

**预期现象**：共享 443 会显著限制端口规则；统计特征 KNN 应明显更好，但 Video 与 Download 仍会出现一定混淆。

## 运行环境说明

| 项目 | 说明 |
|---|---|
| 运行环境 | Kaggle Notebook |
| 计算资源 | CPU（无需 GPU） |
| 网络访问 | 不需要（Internet Off） |
| 数据 | Notebook 内程序生成，无需额外 Dataset |
| 主要依赖 | NumPy、Pandas、scikit-learn、seaborn、Matplotlib |

直接点击 **Run All** 即可完成全部实验。

In [1]:
# 环境准备：Kaggle 默认 Python 环境已包含这些库
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

RANDOM_SEED = 42
LABELS = ['Web', 'DNS', 'Video', 'Download']

print('Environment ready ✅')

## 步骤一：生成“现代端口复用”仿真流量

为了让实验逻辑与现代网络一致，我们不再给四类应用分配彼此完全分离的端口。

### 端口分布设计

| 流量类型 | 端口分布（教学仿真） | 关键现象 |
|---|---|---|
| Web | 80 / **443** / 少量动态端口 | HTTPS 大量使用 443 |
| DNS | 53 / 少量 **443** / 动态端口 | 传统 DNS 与 DoH 场景并存 |
| Video | 主要 **443** / 少量动态端口 | 与 Web 共享 443 |
| Download | 21 / **443** / 动态端口 | 传统 FTP 与 HTTPS 下载并存 |

这样设计后，**端口号本身会发生真实的类别重叠**，尤其是 443。

### 行为统计特征设计

DNS 具有“小包、低字节数、较长间隔”的特征；Video 与 Download 都具有“大包、短间隔、大流量”的特征，因此二者有意保留一定重叠，用来展示混淆矩阵的价值。

> 注意：这些参数是教学用 Synthetic Data，不是从某个真实网络数据集拟合得到的。

In [2]:
# 生成可重复的教学仿真数据
rng = np.random.default_rng(RANDOM_SEED)
N_PER_CLASS = 200


def clipped_normal(mean, std, n, low=1e-6):
    """生成非负连续特征。"""
    return np.clip(rng.normal(mean, std, n), low, None)


def sample_ports(app, n):
    """模拟现代网络中的端口复用：443 同时出现在多个应用中。"""
    u = rng.random(n)
    random_ports = rng.integers(1024, 65536, n)

    if app == 'Web':
        # 30% HTTP/80, 65% HTTPS/443, 5% 动态端口
        return np.where(u < 0.30, 80,
                        np.where(u < 0.95, 443, random_ports)).astype(int)
    if app == 'DNS':
        # 85% DNS/53, 10% DoH-like/443, 5% 动态端口
        return np.where(u < 0.85, 53,
                        np.where(u < 0.95, 443, random_ports)).astype(int)
    if app == 'Video':
        # 90% HTTPS/QUIC-like 443, 10% 动态端口
        return np.where(u < 0.90, 443, random_ports).astype(int)
    if app == 'Download':
        # 25% FTP-like/21, 60% HTTPS/443, 15% 动态端口
        return np.where(u < 0.25, 21,
                        np.where(u < 0.85, 443, random_ports)).astype(int)

    raise ValueError(f'Unknown app: {app}')


# 均值和标准差仅用于构造教学场景；Video 与 Download 刻意设置为部分重叠
PARAMS = {
    'Web': {
        'pkt_len_mean': (700, 170),
        'pkt_len_var':  (50000, 18000),
        'gap_mean':     (0.075, 0.025),
        'gap_var':      (0.030, 0.012),
        'bytes':        (50000, 18000),
    },
    'DNS': {
        'pkt_len_mean': (110, 25),
        'pkt_len_var':  (1200, 450),
        'gap_mean':     (0.350, 0.100),
        'gap_var':      (0.100, 0.030),
        'bytes':        (2500, 700),
    },
    'Video': {
        'pkt_len_mean': (1280, 110),
        'pkt_len_var':  (30000, 10000),
        'gap_mean':     (0.012, 0.004),
        'gap_var':      (0.0045, 0.0015),
        'bytes':        (280000, 70000),
    },
    'Download': {
        'pkt_len_mean': (1380, 90),
        'pkt_len_var':  (22000, 8000),
        'gap_mean':     (0.006, 0.0025),
        'gap_var':      (0.0025, 0.0010),
        'bytes':        (360000, 90000),
    },
}

frames = []
for label_id, app in enumerate(LABELS):
    row = {
        'label': np.full(N_PER_CLASS, label_id, dtype=int),
        'app': np.full(N_PER_CLASS, app),
        'port': sample_ports(app, N_PER_CLASS),
    }
    for feature, (mean, std) in PARAMS[app].items():
        row[feature] = clipped_normal(mean, std, N_PER_CLASS)
    frames.append(pd.DataFrame(row))

df = pd.concat(frames, ignore_index=True)

print(f'Generated {len(df)} flows: {N_PER_CLASS} per class')
display(df.head())

print()
print('Port distribution by application (row-normalized):')
port_group = df['port'].where(df['port'].isin([21, 53, 80, 443]), 'Dynamic')
display(pd.crosstab(df['app'], port_group, normalize='index').round(2))

In [3]:
# 步骤二：在同一测试集上比较“端口规则”与“统计特征 + KNN”
FEATURES = ['pkt_len_mean', 'pkt_len_var', 'gap_mean', 'gap_var', 'bytes']

train_idx, test_idx = train_test_split(
    np.arange(len(df)),
    test_size=0.30,
    random_state=RANDOM_SEED,
    stratify=df['label']
)

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()
y_train = train_df['label']
y_test = test_df['label']

# -------- Baseline：传统端口规则 --------
# 443 本身无法告诉我们是 Web / Video / Download；
# 一个简单的 legacy-style 规则只能固定映射到某个类别，这里映射为 Web。
def legacy_port_rule(port):
    if port == 53:
        return 1  # DNS
    if port in (80, 443):
        return 0  # Web
    if port == 21:
        return 3  # Download
    return -1     # Unknown / unmatched

port_pred = test_df['port'].map(legacy_port_rule).to_numpy()
acc_port = accuracy_score(y_test, port_pred)

# -------- AI 方法：统计特征 + 标准化 + KNN --------
knn_stats = make_pipeline(
    StandardScaler(),
    KNeighborsClassifier(n_neighbors=5)
)
knn_stats.fit(train_df[FEATURES], y_train)
knn_pred = knn_stats.predict(test_df[FEATURES])
acc_stats = accuracy_score(y_test, knn_pred)

# 左图：二维特征空间；右图：准确率对比
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))

for label_id, app in enumerate(LABELS):
    subset = df[df['label'] == label_id]
    axes[0].scatter(
        subset['pkt_len_mean'], subset['gap_mean'],
        label=app, alpha=0.55, s=22
    )

axes[0].set_xlabel('Average Packet Length (bytes)')
axes[0].set_ylabel('Average Inter-arrival Time (s)')
axes[0].set_title('Traffic Feature Space')
axes[0].legend()
axes[0].grid(True, alpha=0.25)

bars = axes[1].bar(['Port Rules', 'Stats + KNN'], [acc_port, acc_stats])
axes[1].set_ylim(0, 1.0)
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Classification Accuracy')
for bar, value in zip(bars, [acc_port, acc_stats]):
    axes[1].text(
        bar.get_x() + bar.get_width()/2,
        value + 0.025,
        f'{value:.1%}',
        ha='center', va='bottom', fontsize=12, fontweight='bold'
    )

plt.tight_layout()
plt.show()

print(f'Port-rule baseline : {acc_port:.1%}')
print(f'Stats + KNN        : {acc_stats:.1%}')
print(f'Improvement        : {(acc_stats - acc_port):.1%}')

# 教学版自检：确保固定随机种子下，实验现象与课程结论一致
assert acc_stats > acc_port + 0.20, 'Synthetic design no longer demonstrates the intended contrast.'
print('Teaching sanity check passed ✅')

## 步骤三：用混淆矩阵深入分析 AI 方法

Accuracy 只能回答“整体有多少预测正确”，但无法回答：

- 哪一类最容易出错？
- 哪两个类别最容易互相混淆？
- 错误是否与特征设计一致？

混淆矩阵中：

- **对角线**：预测正确；
- **非对角线**：真实类别被错分到其他类别；
- 第 *i* 行第 *j* 列：真实为类别 *i*，却预测成类别 *j*。

本实验刻意让 **Video** 与 **Download** 的统计特征部分重叠，因此它们应成为主要混淆来源。

In [4]:
# 混淆矩阵 + 自动找出最明显的错误类别对
cm = confusion_matrix(y_test, knn_pred, labels=range(len(LABELS)))

plt.figure(figsize=(7, 5))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=LABELS, yticklabels=LABELS
)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix: Standardized KNN')
plt.tight_layout()
plt.show()

print('Classification report:')
print(classification_report(y_test, knn_pred, target_names=LABELS, digits=3))

cm_errors = cm.copy()
np.fill_diagonal(cm_errors, 0)
true_id, pred_id = np.unravel_index(np.argmax(cm_errors), cm_errors.shape)
print(
    f'Most frequent error: {LABELS[true_id]} → {LABELS[pred_id]} '
    f'({cm_errors[true_id, pred_id]} samples)'
)

print()
print('课堂互动：')
print('1. 为什么 DNS 几乎不与其他类别混淆？')
print('2. 为什么 Video 与 Download 更容易互混？')
print('3. 如果只能再增加一个统计特征，你会选择什么？')

## 实验结果与分析

### 1. 端口规则为什么明显受限？

本实验中，Web、Video、Download 都大量使用 **443**。端口规则面对 443 时只能固定猜一个类别，因此无法恢复端口背后的应用语义。

这也是本实验最重要的观察：

> **端口号没有消失，但“端口号 = 应用类型”的一一对应关系已经被破坏。**

在固定随机种子下，本 Notebook 的端口规则准确率约为 **50% 左右**。具体数值以你实际 Run All 的输出为准。

### 2. 为什么统计特征 KNN 更好？

KNN 不依赖 443 的应用语义，而是观察 Flow 的行为：包有多大、发得多快、总流量多大、节奏是否稳定。

在本教学仿真中，标准化后的 5 维统计特征可以把多数类别分开，准确率约为 **90% 左右**。

### 3. 为什么不是 100%？

我们有意让 Video 与 Download 的特征区域发生重叠。二者都具有：

- 较大的平均包长；
- 较短的包间隔；
- 较大的总字节数。

因此，混淆矩阵通常会显示 Video ↔ Download 是主要错误来源。

这比“人为让四类完全分开并得到 100%”更适合作为教学实验：它提醒我们，**真实分类问题的价值就在于处理重叠和不确定性。**

### 4. 本实验的局限性

1. 数据是程序生成的 Synthetic Data，不是从真实 pcap 流量提取得到；
2. 参数是为了教学现象而设计，不能解释为真实网络中的统计分布；
3. 测试集与训练集来自同一生成机制，没有体现跨网络、跨时间的 Distribution Shift；
4. 本实验是 Closed-set Classification，没有处理训练集中未出现的新应用。

## 从实验到实际系统

真实网络流量分类还需要考虑更多问题：

- **加密流量**：TLS/QUIC 会隐藏应用层内容，但包长、时序、方向、流持续时间等侧面统计仍可能可用；同时 443 的广泛复用会削弱端口语义。
- **分布漂移（Distribution Shift）**：网络、设备、时间和用户行为变化后，训练分布可能不再匹配测试分布。
- **未知应用（Open-set）**：实际系统会遇到训练阶段未见过的流量，需要“拒识 / 未知类”机制。
- **实时性**：很多统计特征必须观察多个包后才能计算，因此分类精度与决策延迟存在权衡。
- **模型选择**：真实任务可以进一步比较决策树、随机森林、梯度提升、1D-CNN 或序列模型，但模型复杂并不自动意味着更好，仍需基准评测。

---

## 本实验小结

1. **现代网络中，端口仍可观测，但端口与应用不再一一对应。**
2. **流统计特征能够描述应用行为，而不仅是协议字段。**
3. **KNN 是理解“特征空间 + 距离分类”的直观入门模型。**
4. **距离模型必须关注特征尺度，因此标准化是本实验的重要步骤。**
5. **混淆矩阵比单一 Accuracy 更能揭示模型薄弱点。**
6. **教学仿真只能验证原理，真实系统必须在真实数据与分布变化上重新评测。**

---

## 思考与拓展

1. **修改 K 值**：把 `n_neighbors=5` 改为 1、3、10、20，观察性能变化。
2. **去掉 StandardScaler**：比较不标准化时的 KNN，解释为什么距离会被大数值特征主导。
3. **换用决策树**：使用已导入的 `DecisionTreeClassifier`，比较性能和可解释性。
4. **增加噪声 / 重叠**：扩大 Video 与 Download 的标准差，观察混淆矩阵如何变化。
5. **增加新特征**：加入 `packet_count`、`flow_duration` 或上下行字节比，判断是否缓解 Video/Download 混淆。
6. **开放集问题**：人为增加第 5 类“Unknown”，思考普通 KNN 为什么难以拒绝未知样本。

> 下一步可以把本 Demo 从 Synthetic Data 扩展到真实公开流量数据集，形成“仿真理解原理 → 真实数据验证”的两级实验。

---

🏠 [课程主页 · Course Home](https://www.kaggle.com/code/guopingtan/fmi-course-kaggle-hands-on-lab-start-here) &nbsp;|&nbsp; [实验二：自适应视频流与 QoE 优化 →](https://www.kaggle.com/code/guopingtan/fmi-demo2-qoe-optimization)

**FMI Course · Kaggle Hands-on Lab** &nbsp;|&nbsp; MV-AI Lab · Hohai University